# ITCS 6169/8169 — Computer Vision
## Assignment 1: The CNN Challenge
**Due: September 25, 2026 at 11:59 PM EDT**

---

In this assignment, you will build and train a convolutional neural network (CNN) for **16-class scene recognition** using a small dataset of **2,400 training images**.

This notebook is intentionally only a **starter baseline**. It provides a clean training pipeline and a deliberately simple CNN. Your task is to improve the system through informed decisions about architecture, optimization, augmentation, regularization, model selection, and training strategy.

### 2026 workflow: AI is your pair programmer
Use of an AI coding assistant (e.g., ChatGPT, Codex, GitHub Copilot, Claude Code, Gemini, Cursor) is **required**. You may use AI to help implement, debug, refactor, explain, or suggest experiments. However, **you are responsible for the scientific decisions and for verifying all generated code**.

Your final GitHub repository must include an `AI_USAGE.md` file describing representative AI-assisted tasks, at least one incorrect/ineffective AI suggestion, how you verified or corrected it, and one important decision that you made yourself.

### Important model restriction
The primary classifier for Assignment 1 must be a **CNN**. Do **not** use CLIP, DINO/DINOv2, VLM APIs, Vision Transformers as the primary model, or other foundation-model embeddings for your main submission. Pretrained CNNs are allowed if clearly documented.

### Google Colab
Google Colab is recommended, but not required. If using Colab, enable a GPU through **Runtime → Change runtime type → GPU**. You may also use the Educational Cluster or another resource for which you have authorized access.

In [ ]:
# Core imports
from pathlib import Path
import random
import time
import copy

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
# Optional: mount Google Drive when running in Colab.
# Skip this cell if you are running locally or on another server.
try:
    from google.colab import drive
    drive.mount('/content/gdrive')
except ImportError:
    print("Not running in Google Colab; Drive mount skipped.")

## 1. Set your working directory

Download the assignment dataset from the link provided in the assignment handout and organize it so that the notebook can find:

```text
data/
  train/
    class_1/
    ...
  test/
    class_1/
    ...
```

If you use Google Drive, change `PROJECT_ROOT` below to your assignment folder. **Do not hard-code a TA/instructor solution path.**

In [ ]:
# CHANGE THIS PATH to your own assignment directory.
# Example for Colab + Google Drive:
# PROJECT_ROOT = Path('/content/gdrive/MyDrive/ITCS_6169_8169/assignment1')

PROJECT_ROOT = Path('.')
DATA_ROOT = PROJECT_ROOT / 'data'
TRAIN_ROOT = DATA_ROOT / 'train'
TEST_ROOT = DATA_ROOT / 'test'

print('Project root:', PROJECT_ROOT.resolve())
print('Training data:', TRAIN_ROOT)
print('Testing data:', TEST_ROOT)

## 2. Reproducibility and device

A fixed seed makes the starter result easier to reproduce. Exact GPU reproducibility is not always guaranteed across hardware/software versions, so report your environment and seed in the final repository.

In [ ]:
SEED = 0

def set_random_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_random_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

## 3. Loading and preprocessing the data

The starter baseline converts images to grayscale and resizes them to $64\times64$. This is **not necessarily a good final choice**. You should decide whether color, resolution, normalization, and augmentation should be changed.

We split the provided training data into **training and validation sets**. Use the validation set for architecture/hyperparameter decisions. **Do not repeatedly tune on the test set.** The test set should be used for the final evaluation of a selected model.

In [ ]:
IMG_SIZE = 64
BATCH_SIZE = 64
VAL_FRACTION = 0.20

# Deliberately simple preprocessing for the starter baseline.
# TODO: improve this as part of your experiments.
baseline_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

full_train_dataset = datasets.ImageFolder(TRAIN_ROOT, transform=baseline_transform)
test_dataset = datasets.ImageFolder(TEST_ROOT, transform=baseline_transform)

class_names = full_train_dataset.classes
num_classes = len(class_names)
print(f'Classes ({num_classes}): {class_names}')
print(f'Total training images: {len(full_train_dataset)}')
print(f'Test images: {len(test_dataset)}')

val_size = int(round(len(full_train_dataset) * VAL_FRACTION))
train_size = len(full_train_dataset) - val_size
generator = torch.Generator().manual_seed(SEED)
train_dataset, val_dataset = random_split(
    full_train_dataset, [train_size, val_size], generator=generator
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=2, pin_memory=torch.cuda.is_available())
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=2, pin_memory=torch.cuda.is_available())

print(f'Train: {len(train_dataset)} | Validation: {len(val_dataset)} | Test: {len(test_dataset)}')

In [ ]:
# Visualize a few training examples
images, labels = next(iter(train_loader))

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for ax, image, label in zip(axes.flat, images[:8], labels[:8]):
    image = image * 0.5 + 0.5  # unnormalize
    ax.imshow(image.squeeze(0), cmap='gray')
    ax.set_title(class_names[label.item()])
    ax.axis('off')
plt.tight_layout()
plt.show()

## 4. Training a CNN from scratch

Gone are the days when hand-designed features were the default solution. We will start with end-to-end learning and train a CNN directly for 16-way classification.

The architecture below is deliberately weak and simple. **Do not treat it as a recommended architecture.** It exists only to verify that the pipeline works and to give you a baseline to beat.

In [ ]:
class TNet(nn.Module):
    """A deliberately small CNN starter baseline."""
    def __init__(self, num_classes=16):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=3),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=4, stride=4),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16 * 15 * 15, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

model = TNet(num_classes=num_classes)
print(model)
print(f'Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

## 5. Training and evaluation utilities

The training loop below selects the checkpoint with the **best validation accuracy**. This is a much better experimental practice than evaluating the test set after every epoch.

You are welcome to rewrite this code. For example, you may add a scheduler, mixed precision, early stopping, checkpointing, experiment tracking, or richer metrics. If AI helps you implement such changes, verify the implementation and document representative usage in `AI_USAGE.md`.

In [ ]:
@torch.inference_mode()
def evaluate(model, loader, device):
    model.eval()
    correct = 0
    total = 0
    running_loss = 0.0
    criterion = nn.CrossEntropyLoss()

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        logits = model(images)
        loss = criterion(logits, labels)
        running_loss += loss.item() * images.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total


def train_model(model, train_loader, val_loader, optimizer, epochs=20, device=device):
    criterion = nn.CrossEntropyLoss()
    model = model.to(device)
    best_state = copy.deepcopy(model.state_dict())
    best_val_acc = 0.0
    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
    start = time.time()

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        total_seen = 0

        for images, labels in train_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * images.size(0)
            total_seen += labels.size(0)

        train_loss = total_loss / total_seen
        val_loss, val_acc = evaluate(model, val_loader, device)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = copy.deepcopy(model.state_dict())

        elapsed = time.time() - start
        print(f'Epoch {epoch:02d}/{epochs} | '
              f'train loss {train_loss:.4f} | val loss {val_loss:.4f} | '
              f'val acc {val_acc:.4f} | {elapsed:.1f}s')

    model.load_state_dict(best_state)
    print(f'Best validation accuracy: {best_val_acc:.4f}')
    return model, history

In [ ]:
# Train the deliberately simple baseline.
set_random_seed(SEED)
model = TNet(num_classes=num_classes)
optimizer = optim.Adam(model.parameters(), lr=0.002)

model, history = train_model(
    model, train_loader, val_loader, optimizer, epochs=20, device=device
)

In [ ]:
# Plot the starter training history.
plt.figure(figsize=(6, 4))
plt.plot(history['train_loss'], label='Train loss')
plt.plot(history['val_loss'], label='Validation loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

plt.figure(figsize=(6, 4))
plt.plot(history['val_acc'], label='Validation accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

## 6. Your turn: beat the baseline

This simple model should only be treated as a sanity check. Your objective is to substantially improve it.

Rather than asking an AI tool to ``make the accuracy higher,'' formulate hypotheses yourself. Examples of useful questions include whether the model is overfitting, whether color matters, whether stronger augmentation helps, whether a different CNN family is more appropriate, whether pretrained CNN features transfer well, or whether optimization/regularization is limiting performance.

For each important experiment, write down:

| Experiment | What changed? | Why did you try it? | Validation accuracy | What did you learn? |
|---|---|---|---:|---|
| Baseline | Starter TNet | Sanity check | -- | -- |
| Experiment 1 | | | | |
| Experiment 2 | | | | |
| Final model | | | | |

**Do not use the test set to choose among these experiments.**

### Experiment setup

All experiments use `cvexp.py` (in this folder). Its training loop reproduces `train_model` above exactly; for example, TNet at 64×64 with seed 0 gives the same 49.79% validation accuracy as the baseline cell. Each (configuration, seed) pair runs as its own process. The runs are spread over **all visible GPUs**, two at a time per GPU, and each finished run is cached in `results/runs/<experiment>/`, so re-running a cell only trains what is missing. To restrict GPUs, set `CUDA_VISIBLE_DEVICES` before starting Jupyter or pass `gpus=['1', '2']`.

Each configuration is trained with 3 seeds (0, 1, 2) on the **fixed** baseline train/validation split, and we report mean ± std of the best validation accuracy. With only 480 validation images (24–39 per class), single-run differences of 1–2 points are within noise.


In [ ]:
# Shared helpers for the experiments below (implementation in cvexp.py).
import json
from cvexp import run_grid, summarize, DEFAULTS

EXP_SEEDS = [0, 1, 2]
RESULTS_DIR = PROJECT_ROOT / 'results'
RESULTS_DIR.mkdir(exist_ok=True)
print('Baseline config:', DEFAULTS)


### Experiment 1: conventional data augmentation (fixing overfitting)

**Problem.** The baseline overfits strongly: training loss falls to 0.12, while validation loss is lowest around epochs 5–7 and then rises. With only 1,920 training images, TNet memorises the training set.

**Hypothesis.** Label-preserving augmentations enlarge the effective training set and should reduce this gap. Each augmentation below is a plausible invariance for scene images:

| Name | Transform (training only) | Invariance it encodes |
|---|---|---|
| `hflip` | `RandomHorizontalFlip(0.5)` | a mirrored scene is the same scene |
| `translate` | reflect-pad 8 px + `RandomCrop(64)` | small camera shifts |
| `rrc` | `RandomResizedCrop(64, scale=(0.5, 1))` | zoom / framing / mild aspect change |
| `rotate` | `RandomRotation(10)` | slight camera roll |
| `jitter` | `ColorJitter(brightness=0.3, contrast=0.3)` | exposure / lighting |
| `erase` | `RandomErasing(p=0.5)` | partial occlusion |

**Protocol.** Everything else is the baseline (TNet, 64×64 grayscale, Adam 2e-3, 20 epochs, same split). Each augmentation is first tested **alone** (one factor at a time), then the helpful ones are combined. The validation set is never augmented.


In [ ]:
# Visual check: one training image under each augmentation (5 random draws each).
from PIL import Image
from cvexp import build_transform, full_config, AUGMENTATIONS

sample_path, _ = full_train_dataset.samples[train_dataset.indices[0]]
sample = Image.open(sample_path)
names = ['none'] + list(AUGMENTATIONS)
fig, axes = plt.subplots(len(names), 5, figsize=(7, 1.45 * len(names)))
torch.manual_seed(0)
for r, name in enumerate(names):
    tf = build_transform(full_config({'augment': [] if name == 'none' else [name]}), train=True)
    for k in range(5):
        axes[r, k].imshow(tf(sample).squeeze(0) * 0.5 + 0.5, cmap='gray', vmin=0, vmax=1)
        axes[r, k].axis('off')
    axes[r, 0].set_title(name, fontsize=8, loc='left')
plt.suptitle(f'Sample: {Path(sample_path).parent.name}', fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
# Experiment 1a: each augmentation alone vs. the baseline (3 seeds each, all GPUs).
AUG_SINGLE = [[]] + [[a] for a in AUGMENTATIONS]
runs = run_grid('exp1_augmentation', [{'augment': a} for a in AUG_SINGLE], seeds=EXP_SEEDS)

exp1_results = []
for i, aug in enumerate(AUG_SINGLE):
    rs = runs[i * len(EXP_SEEDS):(i + 1) * len(EXP_SEEDS)]
    h = [r['history'] for r in rs]
    exp1_results.append(summarize('+'.join(aug) or 'none (baseline)', rs, {
        'augment': aug, 'histories': h,
        'final_train_loss': float(np.mean([x['train_loss'][-1] for x in h])),
        'min_val_loss': float(np.mean([min(x['val_loss']) for x in h])),
        'final_val_loss': float(np.mean([x['val_loss'][-1] for x in h]))}))
print()
print(f"{'augmentation':18s} {'final train loss':>16s} {'min val loss':>12s} {'final val loss':>14s}")
for row in exp1_results:
    print(f"{row['experiment']:18s} {row['final_train_loss']:16.3f} {row['min_val_loss']:12.3f} {row['final_val_loss']:14.3f}")

with open(RESULTS_DIR / 'exp1_augmentation.json', 'w') as f:
    json.dump(exp1_results, f, indent=2)


In [ ]:
# Experiment 1a: train vs. validation loss (mean over seeds) for each augmentation.
fig, axes = plt.subplots(1, len(exp1_results), figsize=(2.3 * len(exp1_results), 2.6), sharey=True)
for ax, row in zip(axes, exp1_results):
    ax.plot(np.mean([h['train_loss'] for h in row['histories']], 0), label='train')
    ax.plot(np.mean([h['val_loss'] for h in row['histories']], 0), label='val')
    ax.set_title(f"{row['experiment']}\n{row['val_acc_mean']:.1f}±{row['val_acc_std']:.1f}%", fontsize=8)
    ax.set_xlabel('epoch', fontsize=8)
axes[0].set_ylabel('loss'); axes[0].legend(fontsize=7)
plt.tight_layout()
plt.show()


### Experiment 1b: combined augmentation × training length

**Observation from 1a.** Translation, random-resized crop and horizontal flip each reduced the train/validation gap; rotation, jitter and erasing did not. With augmentation the best epoch moved to 16–20 out of 20, so the augmented model is still improving when training stops: it is no longer overfitting but under-trained.

**What changes.** Augmentation = `hflip + rrc + translate` combined, trained for 20, 40, 60 and 100 epochs. As a reference we train **translation alone** for the same epochs, so that a gain from combining can be separated from a gain from simply training longer. Everything else is the baseline (TNet, 64×64 grayscale, Adam 2e-3, constant learning rate, same split, 3 seeds).


In [ ]:
# Experiment 1b: {translate} vs. {hflip + rrc + translate}, each at 20 / 40 / 60 / 100 epochs.
AUG_SETS = {'translate': ['translate'], 'hflip+rrc+translate': ['hflip', 'rrc', 'translate']}
EPOCHS = [20, 40, 60, 100]
grid = [(name, ep) for name in AUG_SETS for ep in EPOCHS]
runs = run_grid('exp1b_combined_epochs', [{'augment': AUG_SETS[n], 'epochs': ep} for n, ep in grid],
                seeds=EXP_SEEDS)

exp1b_results = []
for i, (name, ep) in enumerate(grid):
    rs = runs[i * len(EXP_SEEDS):(i + 1) * len(EXP_SEEDS)]
    h = [r['history'] for r in rs]
    exp1b_results.append(summarize(f'{name} @ {ep} ep', rs, {
        'augment': AUG_SETS[name], 'epochs': ep, 'histories': h,
        'final_train_loss': float(np.mean([x['train_loss'][-1] for x in h])),
        'final_val_loss': float(np.mean([x['val_loss'][-1] for x in h])),
        # The best epoch is a max over a noisy curve, which favours longer runs; the mean over
        # the last 10 epochs is a less optimistic check that the trend is real.
        'val_acc_last10': float(np.mean([np.mean(x['val_acc'][-10:]) for x in h]) * 100)}))
print()
print(f"{'setting':32s} {'best val acc':>12s} {'last-10 val acc':>15s} {'final train loss':>16s} {'final val loss':>14s}")
for row in exp1b_results:
    print(f"{row['experiment']:32s} {row['val_acc_mean']:12.2f} {row['val_acc_last10']:15.2f} "
          f"{row['final_train_loss']:16.3f} {row['final_val_loss']:14.3f}")

with open(RESULTS_DIR / 'exp1b_combined_epochs.json', 'w') as f:
    json.dump(exp1b_results, f, indent=2)


In [ ]:
# Experiment 1b: 100-epoch curves (mean over seeds) for both augmentation sets.
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
for row in [r for r in exp1b_results if r['epochs'] == 100]:
    h = row['histories']
    va = np.mean([x['val_acc'] for x in h], 0)
    line, = axes[0].plot(range(1, 101), va, label=row['experiment'])
    axes[1].plot(range(1, 101), np.mean([x['train_loss'] for x in h], 0), color=line.get_color(),
                 label=f"{row['experiment']} train")
    axes[1].plot(range(1, 101), np.mean([x['val_loss'] for x in h], 0), '--', color=line.get_color(),
                 label=f"{row['experiment']} val")
axes[0].set_xlabel('epoch'); axes[0].set_ylabel('validation accuracy'); axes[0].legend(fontsize=8)
axes[1].set_xlabel('epoch'); axes[1].set_ylabel('loss'); axes[1].legend(fontsize=7)
plt.tight_layout()
plt.show()


### Experiment 1c: does the combined-augmentation model converge? (up to 600 epochs)

**Question.** At 100 epochs, the training loss of the combined-augmentation model is still falling slowly. Is the model still under-trained, or has validation accuracy levelled off?

**What changes.** Same setting as the best 1b run (`hflip + rrc + translate`, constant learning rate 2e-3), but trained for 600 epochs. With a constant learning rate and fixed seeds, the first *N* epochs of this run are identical to an *N*-epoch run; we check this against the 100-epoch 1b run. So one 600-epoch curve also gives the 200-, 300-, 400- and 500-epoch results without separate runs.

**Reporting.** Max-over-epochs validation accuracy becomes more optimistic the longer the run, because it is the maximum of more noisy values. So we also report the mean accuracy over a 10-epoch window ending at each checkpoint.


In [ ]:
# Experiment 1c: combined augmentation for 600 epochs (400 and 500 are read off the same curve).
COMBO = ['hflip', 'rrc', 'translate']
runs_600 = run_grid('exp1c_long', [{'augment': COMBO, 'epochs': 600}], seeds=EXP_SEEDS)
hist = {k: np.array([r['history'][k] for r in runs_600]) for k in ['train_loss', 'val_loss', 'val_acc']}

# Sanity check: the first 100 epochs must equal the 100-epoch run of Experiment 1b.
runs_100 = run_grid('exp1b_combined_epochs', [{'augment': COMBO, 'epochs': 100}], seeds=EXP_SEEDS)
same = all(r5['history']['val_acc'][:100] == r1['history']['val_acc'] for r5, r1 in zip(runs_600, runs_100))
print('First 100 epochs identical to the 100-epoch run:', same)

exp1c_rows = []
print(f"\n{'epochs':>6s} {'best val acc (max)':>20s} {'val acc, last 10 ep':>20s} {'train loss':>10s} {'val loss':>9s}")
for n in [100, 200, 300, 400, 500, 600]:
    best = hist['val_acc'][:, :n].max(1) * 100
    last10 = hist['val_acc'][:, n - 10:n].mean(1) * 100
    row = {'epochs': n, 'best_mean': best.mean(), 'best_std': best.std(),
           'last10_mean': last10.mean(), 'last10_std': last10.std(),
           'train_loss': hist['train_loss'][:, n - 1].mean(), 'val_loss': hist['val_loss'][:, n - 1].mean()}
    exp1c_rows.append(row)
    print(f"{n:6d} {row['best_mean']:13.2f} ± {row['best_std']:4.2f} {row['last10_mean']:13.2f} ± {row['last10_std']:4.2f} "
          f"{row['train_loss']:10.3f} {row['val_loss']:9.3f}")
print('best epochs (600-epoch runs):', [r['best_epoch'] for r in runs_600])

with open(RESULTS_DIR / 'exp1c_long.json', 'w') as f:
    json.dump({'rows': exp1c_rows, 'best_epochs': [r['best_epoch'] for r in runs_600]}, f, indent=2, default=float)


In [ ]:
# Experiment 1c: 600-epoch curves (mean over seeds; accuracy smoothed over 10 epochs).
ep = np.arange(1, 601)
smooth = lambda x: np.convolve(x, np.ones(10) / 10, mode='valid')
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
axes[0].plot(ep, hist['val_acc'].mean(0), alpha=0.3, color='tab:orange', label='val acc (raw)')
axes[0].plot(ep[9:], smooth(hist['val_acc'].mean(0)), color='tab:orange', label='val acc (10-ep mean)')
axes[0].set_xlabel('epoch'); axes[0].set_ylabel('validation accuracy'); axes[0].legend(fontsize=8)
axes[1].plot(ep, hist['train_loss'].mean(0), label='train')
axes[1].plot(ep, hist['val_loss'].mean(0), label='val')
axes[1].set_xlabel('epoch'); axes[1].set_ylabel('loss'); axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()


### Experiment 2: learning-rate schedule

**Observation from 1c.** With a constant learning rate, the combined-augmentation model gains most of its accuracy by epoch ~200 but keeps creeping up until epoch 600, and its best epochs fall near the end of the run. This is the usual sign that the step size stays too large for the optimiser to settle into a minimum.

**Hypothesis.** A large learning rate early (fast progress) followed by a decay (fine convergence) should reach the 600-epoch constant-lr accuracy in far fewer epochs.

**What changes.** Only the schedule (and, in one sub-experiment, the peak lr). Each schedule is applied per iteration as a multiplier of the peak lr = 2e-3:

| Schedule | Shape |
|---|---|
| `constant` | 2e-3 throughout (reference = Experiment 1c) |
| `step` | ×0.1 at 50% and again at 75% of training |
| `exp` | exponential decay to 1% of the peak at the end |
| `cosine` | half-cosine from the peak to 0 |
| `warmup_cosine` | linear warm-up over 5 epochs, then cosine to 0 |
| `onecycle` | `OneCycleLR`: warm-up to the peak over the first 30%, then cosine to ~0 |

Each schedule is trained for **100 and 200 epochs**. A schedule's shape depends on the total length, so unlike Experiment 1c, shorter runs cannot be read off a longer one. We also test whether a decaying schedule tolerates a **larger peak lr** (cosine with 5e-3 and 1e-2, 200 epochs). Everything else = best setting so far (TNet, 64×64 grayscale, `hflip + rrc + translate`, Adam, same split, 3 seeds).

**Metric.** Decaying schedules end with a stable model, so besides best-epoch validation accuracy we report the mean over the last 10 epochs (the accuracy you would get without picking the best epoch).


In [ ]:
# Experiment 2: lr schedules x {100, 200} epochs, plus cosine with a larger peak lr.
from cvexp import SCHEDULES
COMBO = ['hflip', 'rrc', 'translate']
grid = [(sch, ep, 0.002) for ep in [100, 200] for sch in SCHEDULES] + \
       [('cosine', 200, 0.005), ('cosine', 200, 0.01)]
runs = run_grid('exp2_schedule', [{'augment': COMBO, 'schedule': sch, 'epochs': ep, 'lr': lr}
                                  for sch, ep, lr in grid], seeds=EXP_SEEDS)

# Sanity check: constant lr @ 200 epochs must equal the first 200 epochs of the 600-epoch run (Exp. 1c).
const200 = [r for r in runs if r['config']['schedule'] == 'constant' and r['config']['epochs'] == 200]
print('constant@200 identical to first 200 epochs of Exp. 1c:',
      all(a['history']['val_acc'] == b['history']['val_acc'][:200] for a, b in zip(const200, runs_600)))

exp2_results = []
for i, (sch, ep, lr) in enumerate(grid):
    rs = runs[i * len(EXP_SEEDS):(i + 1) * len(EXP_SEEDS)]
    h = [r['history'] for r in rs]
    exp2_results.append(summarize(f'{sch} lr={lr:g} @ {ep} ep', rs, {
        'schedule': sch, 'epochs': ep, 'lr': lr, 'histories': h,
        'val_acc_last10': float(np.mean([np.mean(x['val_acc'][-10:]) for x in h]) * 100),
        'val_acc_last10_std': float(np.std([np.mean(x['val_acc'][-10:]) for x in h]) * 100),
        'final_train_loss': float(np.mean([x['train_loss'][-1] for x in h])),
        'final_val_loss': float(np.mean([x['val_loss'][-1] for x in h]))}))
print()
print(f"{'setting':30s} {'best val acc':>13s} {'last-10 val acc':>16s} {'train loss':>10s} {'val loss':>9s}")
for row in exp2_results:
    print(f"{row['experiment']:30s} {row['val_acc_mean']:7.2f} ± {row['val_acc_std']:4.2f} "
          f"{row['val_acc_last10']:9.2f} ± {row['val_acc_last10_std']:4.2f} "
          f"{row['final_train_loss']:10.3f} {row['final_val_loss']:9.3f}")

with open(RESULTS_DIR / 'exp2_schedule.json', 'w') as f:
    json.dump(exp2_results, f, indent=2)


In [ ]:
# Experiment 2: lr trace and validation accuracy (mean over seeds) for each 200-epoch schedule at lr=2e-3.
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for row in [r for r in exp2_results if r['epochs'] == 200 and r['lr'] == 0.002]:
    h = row['histories']
    axes[0].plot(range(1, 201), h[0]['lr'], label=row['schedule'])
    va = np.mean([x['val_acc'] for x in h], 0)
    axes[1].plot(range(10, 201), np.convolve(va, np.ones(10) / 10, mode='valid'),
                 label=f"{row['schedule']} ({row['val_acc_last10']:.1f}%)")
axes[0].set_xlabel('epoch'); axes[0].set_ylabel('learning rate'); axes[0].legend(fontsize=8)
axes[1].set_xlabel('epoch'); axes[1].set_ylabel('val accuracy (10-epoch mean)'); axes[1].legend(fontsize=8)
axes[1].set_ylim(0.4, 0.72)
plt.tight_layout()
plt.show()


### Experiment 3: input resolution, revisited

**Why again.** The preliminary sweep (no augmentation, constant lr, 20 epochs) found that 256×256 inputs helped (+5.3 points), but every larger input overfit even more (training loss → 0.003). Now that augmentation and the cosine schedule control overfitting, and TNet is under-fitting rather than overfitting at 64×64, larger inputs may help more (more detail, and a larger classifier).

**What changes.** Only `img_size` ∈ {64, 200, 224, 256}; everything else is the best setting from Experiment 2 (`hflip + rrc + translate`, cosine lr with peak 5e-3, 200 epochs, Adam, same split, 3 seeds). The translation padding scales with the input (`s // 8`, i.e. 8 px at 64), so translation is equally strong at every size. Random-resized crop and flip do not depend on the input size.

**Confound (as before).** TNet's linear layer grows with the input (58K → 1.0M parameters), so a gain mixes more pixels with more classifier parameters.


In [ ]:
# Experiment 3: input resolution with the best training setting so far.
BEST = {'augment': ['hflip', 'rrc', 'translate'], 'schedule': 'cosine', 'lr': 0.005, 'epochs': 200}
SIZES = [64, 200, 224, 256]
runs = run_grid('exp3_resolution', [{**BEST, 'img_size': s} for s in SIZES], seeds=EXP_SEEDS)

exp3_results = []
for i, size in enumerate(SIZES):
    rs = runs[i * len(EXP_SEEDS):(i + 1) * len(EXP_SEEDS)]
    h = [r['history'] for r in rs]
    last10 = np.array([np.mean(x['val_acc'][-10:]) for x in h]) * 100
    exp3_results.append(summarize(f'TNet @ {size}x{size}', rs, {
        'img_size': size, 'histories': h,
        'val_acc_last10': float(last10.mean()), 'val_acc_last10_std': float(last10.std()),
        'final_train_loss': float(np.mean([x['train_loss'][-1] for x in h])),
        'final_val_loss': float(np.mean([x['val_loss'][-1] for x in h]))}))
print()
print(f"{'input':14s} {'params':>10s} {'best val acc':>13s} {'last-10 val acc':>16s} {'train loss':>10s} {'val loss':>9s} {'s/run':>6s}")
for row in exp3_results:
    print(f"{row['img_size']:>4d}x{row['img_size']:<9d} {row['params']:10,d} {row['val_acc_mean']:7.2f} ± {row['val_acc_std']:4.2f} "
          f"{row['val_acc_last10']:9.2f} ± {row['val_acc_last10_std']:4.2f} {row['final_train_loss']:10.3f} "
          f"{row['final_val_loss']:9.3f} {row['time_s_per_run']:6.0f}")

with open(RESULTS_DIR / 'exp3_resolution.json', 'w') as f:
    json.dump(exp3_results, f, indent=2)


In [ ]:
# Experiment 3: validation accuracy (10-epoch mean) and train/val loss per resolution (mean over seeds).
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for row in exp3_results:
    h = row['histories']
    va = np.mean([x['val_acc'] for x in h], 0)
    line, = axes[0].plot(range(10, 201), np.convolve(va, np.ones(10) / 10, mode='valid'),
                         label=f"{row['img_size']}px ({row['val_acc_last10']:.1f}%)")
    axes[1].plot(np.mean([x['train_loss'] for x in h], 0), color=line.get_color(), label=f"{row['img_size']}px train")
    axes[1].plot(np.mean([x['val_loss'] for x in h], 0), '--', color=line.get_color())
axes[0].set_xlabel('epoch'); axes[0].set_ylabel('val accuracy (10-epoch mean)'); axes[0].legend(fontsize=8)
axes[1].set_xlabel('epoch'); axes[1].set_ylabel('loss (solid train, dashed val)'); axes[1].legend(fontsize=7)
axes[1].set_ylim(0.8, 2.5)   # the 256px run starts at a loss of ~27 (see the text)
plt.tight_layout()
plt.show()


### Experiment 4: deeper CNN architectures (from scratch)

**Observation from Experiments 2–3.** With augmentation and a cosine schedule, TNet's training loss stays near 1.0: the one-layer network can no longer fit the augmented training data, so it is **capacity-limited** (under-fitting). Its classifier also grows with the input size, which confounded Experiment 3.

**Hypothesis.** A deeper CNN with a resolution-independent head (global average pooling) learns hierarchical features (edges → textures → parts → scene layout) and should break this ceiling, especially on the hard indoor classes.

**What changes.** Only the architecture (plus its learning rate, see below). All models are standard torchvision CNNs, trained **from scratch** (`weights=None`). ImageNet initialisation is a separate, later experiment.

| Model | Params | Family |
|---|---:|---|
| ResNet-18 / 34 / 50 | 11.2M / 21.3M / 23.5M | residual (basic / bottleneck blocks) |
| DenseNet-121 | 7.0M | dense connectivity |
| ConvNeXt-Tiny | 27.8M | modernised ResNet (7×7 depthwise conv, LayerNorm, GELU) |
| MobileNetV3-Large | 4.2M | efficient, depthwise separable + squeeze-excitation |
| EfficientNet-B0 | 4.0M | efficient, compound-scaled MBConv |

**Fair comparison.** The peak lr 5e-3 was tuned for TNet; deep networks with Adam (ConvNeXt especially) often need less. To avoid penalising an architecture with the wrong lr, each model is trained with peak lr **1e-3 and 5e-3**, and we keep the better one per model (chosen on validation, never on test). Everything else is the best setting so far: 200×200 grayscale (replicated to 3 channels), `hflip + rrc + translate`, cosine schedule, 200 epochs, batch 64, Adam, same split, 3 seeds. bf16 mixed precision is used for speed.


In [ ]:
# Experiment 4: standard CNNs from scratch x peak lr {1e-3, 5e-3}.
from cvexp import TORCHVISION_MODELS
DEEP_BASE = {'augment': ['hflip', 'rrc', 'translate'], 'schedule': 'cosine', 'epochs': 200,
             'img_size': 200, 'amp': True}
LRS = [0.001, 0.005]
grid = [(m, lr) for m in TORCHVISION_MODELS for lr in LRS]
runs = run_grid('exp4_architecture', [{**DEEP_BASE, 'model': m, 'lr': lr} for m, lr in grid], seeds=EXP_SEEDS)

exp4_results = []
for i, (m, lr) in enumerate(grid):
    rs = runs[i * len(EXP_SEEDS):(i + 1) * len(EXP_SEEDS)]
    h = [r['history'] for r in rs]
    last10 = np.array([np.mean(x['val_acc'][-10:]) for x in h]) * 100
    exp4_results.append(summarize(f'{m} lr={lr:g}', rs, {
        'model': m, 'lr': lr, 'histories': h,
        'val_acc_last10': float(last10.mean()), 'val_acc_last10_std': float(last10.std()),
        'final_train_loss': float(np.mean([x['train_loss'][-1] for x in h])),
        'final_val_loss': float(np.mean([x['val_loss'][-1] for x in h]))}))

# Per model, keep the lr with the higher last-10 validation accuracy.
best_per_model = {}
for row in exp4_results:
    if row['model'] not in best_per_model or row['val_acc_last10'] > best_per_model[row['model']]['val_acc_last10']:
        best_per_model[row['model']] = row
print()
print(f"{'model':20s} {'params':>8s} {'lr':>6s} {'best val acc':>13s} {'last-10':>13s} {'train loss':>10s} {'val loss':>9s} {'min/run':>8s}")
for m, row in sorted(best_per_model.items(), key=lambda kv: -kv[1]['val_acc_last10']):
    print(f"{m:20s} {row['params'] / 1e6:7.1f}M {row['lr']:6g} {row['val_acc_mean']:7.2f} ± {row['val_acc_std']:4.2f} "
          f"{row['val_acc_last10']:7.2f} ± {row['val_acc_last10_std']:4.2f} {row['final_train_loss']:10.3f} "
          f"{row['final_val_loss']:9.3f} {row['time_s_per_run'] / 60:8.1f}")

with open(RESULTS_DIR / 'exp4_architecture.json', 'w') as f:
    json.dump(exp4_results, f, indent=2)


In [ ]:
# Experiment 4: accuracy vs. parameters, and validation curves of each model at its better lr.
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for m, row in best_per_model.items():
    axes[0].errorbar(row['params'] / 1e6, row['val_acc_last10'], yerr=row['val_acc_last10_std'], fmt='o', capsize=3)
    axes[0].annotate(m, (row['params'] / 1e6, row['val_acc_last10']), fontsize=7, xytext=(4, 2), textcoords='offset points')
    va = np.mean([x['val_acc'] for x in row['histories']], 0)
    axes[1].plot(range(10, 201), np.convolve(va, np.ones(10) / 10, mode='valid'), label=f"{m} (lr {row['lr']:g})")
tnet = exp3_results[1]   # TNet @ 200x200 from Experiment 3
axes[0].axhline(tnet['val_acc_last10'], ls='--', color='gray', label='TNet @ 200 (Exp. 3)')
axes[0].set_xlabel('parameters (M)'); axes[0].set_ylabel('val accuracy, last-10 (%)'); axes[0].legend(fontsize=7)
axes[1].set_xlabel('epoch'); axes[1].set_ylabel('val accuracy (10-epoch mean)'); axes[1].legend(fontsize=7)
plt.tight_layout()
plt.show()


### Experiment 5: keep the colour of originally-colour images (colour-shortcut test)

**Background (data analysis).** Only one class, **Flower**, is stored in colour (RGB); all other 15 classes are grayscale. So far every image has been converted to grayscale, which removes this cue.

**Question.** Should the originally-colour images keep their colour? It is real information, but because colour appears *only* in Flower, a model can learn the shortcut "colour ⇒ Flower" instead of recognising flowers. Such a model would look perfect on this test set (where Flower is also the only colour class) but would fail on gray flowers, or on colour images of other scenes.

**What changes.** `keep_color=True`: RGB images stay RGB, grayscale images are replicated to 3 channels (so non-Flower inputs are identical in both arms). Everything else is the best Experiment 4 setting. We test the two best architectures, EfficientNet-B0 (lr 5e-3) and ResNet-50 (lr 1e-3), with 3 seeds each.

**Shortcut probes** (final-epoch model, validation set):
1. overall and **Flower** accuracy, gray vs. colour model;
2. **Flower accuracy when the Flower images are converted to gray** at test time: a large drop for the colour model means it relies on colour rather than flower shape and texture;
3. **false-Flower rate**: how often a non-Flower (gray) image is predicted as Flower.


In [ ]:
# Experiment 5: grayscale everywhere vs. keep colour of originally-RGB images.
COLOR_ARCHS = {'efficientnet_b0': 0.005, 'resnet50': 0.001}   # best lr per model from Experiment 4
grid = [(m, kc) for m in COLOR_ARCHS for kc in [False, True]]
runs = run_grid('exp5_color', [{**DEEP_BASE, 'model': m, 'lr': COLOR_ARCHS[m], 'keep_color': kc}
                               for m, kc in grid], seeds=EXP_SEEDS)

# Sanity check: the gray arm must reproduce Experiment 4 exactly (same config, re-run with probes).
from cvexp import config_id
for m in COLOR_ARCHS:
    gray = [r for r in runs if r['config']['model'] == m and not r['config']['keep_color']]
    old = [json.loads((RESULTS_DIR / 'runs' / 'exp4_architecture' /
                       f"{config_id({**DEEP_BASE, 'model': m, 'lr': COLOR_ARCHS[m]})}_s{s}.json").read_text())
           for s in EXP_SEEDS]
    print(f'{m}: gray arm identical to Experiment 4:',
          all(a['history']['val_acc'] == b['history']['val_acc'] for a, b in zip(gray, old)))

FLOWER = runs[0]['classes'].index('Flower')

def probes(r):
    y, p, pg = (np.array(r[k]) for k in ['val_labels', 'val_preds', 'val_preds_gray'])
    fl = y == FLOWER
    return {'overall': (p == y).mean() * 100,
            'flower': (p[fl] == FLOWER).mean() * 100,
            'flower_as_gray': (pg[fl] == FLOWER).mean() * 100,
            'false_flower': (p[~fl] == FLOWER).mean() * 100,
            'non_flower': (p[~fl] == y[~fl]).mean() * 100}

exp5_results = []
print(f"\n{'model':16s} {'colour':>6s} {'last-10':>8s} {'final acc':>10s} {'Flower':>7s} "
      f"{'Flower->gray':>13s} {'false-Flower':>13s} {'non-Flower acc':>15s}")
for i, (m, kc) in enumerate(grid):
    rs = runs[i * len(EXP_SEEDS):(i + 1) * len(EXP_SEEDS)]
    pr = [probes(r) for r in rs]
    row = {'model': m, 'keep_color': kc,
           'val_acc_last10': float(np.mean([np.mean(r['history']['val_acc'][-10:]) for r in rs]) * 100),
           **{k: float(np.mean([x[k] for x in pr])) for k in pr[0]},
           **{k + '_std': float(np.std([x[k] for x in pr])) for k in pr[0]}}
    exp5_results.append(row)
    print(f"{m:16s} {('keep' if kc else 'gray'):>6s} {row['val_acc_last10']:8.2f} {row['overall']:10.2f} "
          f"{row['flower']:7.1f} {row['flower_as_gray']:13.1f} {row['false_flower']:13.2f} {row['non_flower']:15.2f}")
print(f"(Flower has {int((np.array(runs[0]['val_labels']) == FLOWER).sum())} validation images; "
      f"one image = {100 / (np.array(runs[0]['val_labels']) == FLOWER).sum():.1f} points of Flower accuracy)")

with open(RESULTS_DIR / 'exp5_color.json', 'w') as f:
    json.dump(exp5_results, f, indent=2)


### Experiment 6: multi-task learning with a self-supervised auxiliary task

**Motivation.** The deep CNNs overfit again (training loss ≈ 0.01 vs. 0.8–1.0 on validation): with 1,920 labelled images, the classification loss alone lets the network memorise. A self-supervised (SSL) auxiliary task asks the same backbone to also solve a label-free task on the same images, which can act as a regulariser and push it towards more general features.

**Design.** Total loss = cross-entropy + λ · SSL loss, computed on the **training images only** (`ssl_aux.py`). The validation and test images are never used, so validation stays a clean selection set and the results are comparable with Experiments 1–5. The SSL heads are used only during training: the deployed classifier is the unchanged EfficientNet-B0.

| Method | Auxiliary task |
|---|---|
| `rotation` | predict the rotation (0/90/180/270°) of a rotated copy |
| `simclr` | contrastive (NT-Xent, τ = 0.2) between two augmented views |
| `simsiam` | BYOL-style: predict one view's embedding from the other (stop-gradient, no negatives) |
| `mim` | masked image modelling, MAE-style for CNNs: mask 60% of 20×20 patches, reconstruct them with a small conv decoder |

The SSL losses have different scales, so each method is run with **λ ∈ {0.1, 1.0}**; we keep the better λ per method on validation. Everything else = Experiment 4's best: EfficientNet-B0 from scratch, lr 5e-3, 200×200 gray, `hflip + rrc + translate`, cosine schedule, 200 epochs, 3 seeds. The no-SSL reference is the Experiment 4 run.


In [ ]:
# Experiment 6: classification + SSL auxiliary task (training images only).
SSL_BASE = {**DEEP_BASE, 'model': 'efficientnet_b0', 'lr': 0.005}
SSL_GRID = [(m, w) for m in ['rotation', 'simclr', 'simsiam', 'mim'] for w in [0.1, 1.0]]
runs = run_grid('exp6_ssl', [{**SSL_BASE, 'ssl': m, 'ssl_weight': w} for m, w in SSL_GRID], seeds=EXP_SEEDS)

# Reference: EfficientNet-B0 without SSL (Experiment 4, cached).
ref_runs = run_grid('exp4_architecture', [SSL_BASE], seeds=EXP_SEEDS)

def ssl_row(name, rs, extra):
    h = [r['history'] for r in rs]
    last10 = np.array([np.mean(x['val_acc'][-10:]) for x in h]) * 100
    return summarize(name, rs, {**extra,
        'val_acc_last10': float(last10.mean()), 'val_acc_last10_std': float(last10.std()),
        'final_train_loss': float(np.mean([x['train_loss'][-1] for x in h])),
        'final_val_loss': float(np.mean([x['val_loss'][-1] for x in h])),
        'final_ssl_loss': float(np.mean([x['ssl_loss'][-1] for x in h])) if 'ssl_loss' in h[0] else None})

exp6_results = [ssl_row('none (Exp. 4)', ref_runs, {'ssl': 'none', 'ssl_weight': 0.0})]
for i, (m, w) in enumerate(SSL_GRID):
    exp6_results.append(ssl_row(f'{m} λ={w:g}', runs[i * len(EXP_SEEDS):(i + 1) * len(EXP_SEEDS)],
                                {'ssl': m, 'ssl_weight': w}))
print()
print(f"{'setting':16s} {'best val acc':>13s} {'last-10':>13s} {'train CE':>9s} {'val loss':>9s} {'SSL loss':>9s} {'min/run':>8s}")
for row in exp6_results:
    ssl_l = '' if row['final_ssl_loss'] is None else f"{row['final_ssl_loss']:9.3f}"
    print(f"{row['experiment']:16s} {row['val_acc_mean']:7.2f} ± {row['val_acc_std']:4.2f} "
          f"{row['val_acc_last10']:7.2f} ± {row['val_acc_last10_std']:4.2f} {row['final_train_loss']:9.3f} "
          f"{row['final_val_loss']:9.3f} {ssl_l:>9s} {row['time_s_per_run'] / 60:8.1f}")

with open(RESULTS_DIR / 'exp6_ssl.json', 'w') as f:
    json.dump([{k: v for k, v in r.items()} for r in exp6_results], f, indent=2)


### Experiment 7: SSL auxiliary task on the (unlabelled) validation images

**Idea.** Same multi-task setup as Experiment 6, but the SSL loss is computed on batches of **validation images** (labels never read), while classification still uses the labelled training images. This uses 480 extra images that the classifier otherwise never sees.

**Caveat: validation is no longer a clean selection set for these runs.** The models adapt their features to exactly the images they are evaluated on, so their validation accuracy is optimistic and **not comparable** with Experiments 1–6. These runs are therefore judged on the test set (next to the same SSL methods trained on the training images), and they are **exploratory**: they are not used to choose the final model, which is fixed beforehand on validation (plain EfficientNet-B0).

Each method uses its better λ from Experiment 6 (rotation 1, SimCLR 0.1, SimSiam 0.1, MIM 1); everything else is the Experiment 6 setting. Checkpoints are saved for the test evaluation below.


In [ ]:
# Experiment 7: SSL on the unlabelled validation images (exploratory; validation accuracy is optimistic).
SSL_BEST_LAMBDA = {'rotation': 1.0, 'simclr': 0.1, 'simsiam': 0.1, 'mim': 1.0}   # from Experiment 6
exp7_cfgs = {m: {**SSL_BASE, 'ssl': m, 'ssl_weight': w, 'ssl_source': 'val'} for m, w in SSL_BEST_LAMBDA.items()}
runs7 = run_grid('exp7_ssl_val', list(exp7_cfgs.values()), seeds=EXP_SEEDS, save_checkpoint=True)

exp7_results = []
for i, m in enumerate(exp7_cfgs):
    rs = runs7[i * len(EXP_SEEDS):(i + 1) * len(EXP_SEEDS)]
    exp7_results.append(ssl_row(f'{m} λ={SSL_BEST_LAMBDA[m]:g} on val', rs, {'ssl': m, 'ssl_source': 'val'}))
print('\n(validation accuracy below is optimistic: the SSL loss was trained on these images)')
for row in exp7_results:
    print(f"{row['experiment']:24s} last-10 val {row['val_acc_last10']:6.2f} ± {row['val_acc_last10_std']:4.2f}  "
          f"final SSL loss {row['final_ssl_loss']:7.3f}  {row['time_s_per_run'] / 60:5.1f} min/run")


### Summary table (Table 1 of the report)

Rebuilds the report's experiment table from the cached runs: best-epoch and last-10-epoch validation accuracy (mean ± std over 3 seeds) for every row.


In [ ]:
# Report Table 1: best-epoch and last-10-epoch validation accuracy for each experiment row.
from cvexp import config_id

def acc_stats(exp_name, cfg, upto=None):
    best, last10 = [], []
    for s in EXP_SEEDS:
        va = json.loads((RESULTS_DIR / 'runs' / exp_name / f'{config_id(cfg)}_s{s}.json').read_text())['history']['val_acc']
        va = va[:upto] if upto else va
        best.append(max(va) * 100)
        last10.append(np.mean(va[-10:]) * 100)
    return np.mean(best), np.std(best), np.mean(last10), np.std(last10)

COMBO = ['hflip', 'rrc', 'translate']
TABLE = [
    ('Starter TNet (64², gray, 20 ep.)', 'exp1_augmentation', {'augment': []}, None),
    ('  + horizontal flip', 'exp1_augmentation', {'augment': ['hflip']}, None),
    ('  + translation (pad 8, crop)', 'exp1_augmentation', {'augment': ['translate']}, None),
    ('  + random-resized crop', 'exp1_augmentation', {'augment': ['rrc']}, None),
    ('  + rotation', 'exp1_augmentation', {'augment': ['rotate']}, None),
    ('  + jitter', 'exp1_augmentation', {'augment': ['jitter']}, None),
    ('  + erasing', 'exp1_augmentation', {'augment': ['erase']}, None),
    ('+ translation, 100 ep. (1b)', 'exp1b_combined_epochs', {'augment': ['translate'], 'epochs': 100}, None),
    ('+ flip+RRC+translation, 100 ep.', 'exp1b_combined_epochs', {'augment': COMBO, 'epochs': 100}, None),
    ('  same, 200 ep. (1c)', 'exp1c_long', {'augment': COMBO, 'epochs': 600}, 200),
    ('  same, 600 ep. (1c)', 'exp1c_long', {'augment': COMBO, 'epochs': 600}, None),
    ('+ cosine lr, 200 ep. (2)', 'exp2_schedule', {'augment': COMBO, 'schedule': 'cosine', 'epochs': 200}, None),
    ('  + peak lr 5e-3', 'exp2_schedule', {'augment': COMBO, 'schedule': 'cosine', 'epochs': 200, 'lr': 0.005}, None),
] + [(f'  input {s}x{s} (3)', 'exp3_resolution',
      {'augment': COMBO, 'schedule': 'cosine', 'lr': 0.005, 'epochs': 200, 'img_size': s}, None)
     for s in [200, 224, 256]]
print(f"{'experiment':34s} {'best epoch':>13s} {'last-10':>13s}")
for label, exp_name, cfg, upto in TABLE:
    b, bs, l, ls = acc_stats(exp_name, cfg, upto)
    print(f"{label:34s} {b:6.1f} ± {bs:3.1f}  {l:6.1f} ± {ls:3.1f}")


### Preliminary run: input resolution

*Run before addressing overfitting. Because every larger input overfit even more, we postpone the resolution study until regularisation is in place (it will be redone as a later experiment).*

**Hypothesis.** The starter resizes every image to $64\times64$, but the median image is $256\times256$ and the shortest side is never below 200 px (see the data analysis in the report). Down-sampling by ~4× discards most of the texture and layout detail that distinguishes scenes, so a higher input resolution should improve accuracy.

**What changes.** Only `IMG_SIZE` ∈ {64 (reference), 200, 224, 256}. Preprocessing (grayscale + `Resize((s, s))`), optimiser, learning rate, epochs and the train/validation split are identical to the baseline.

**Confound to keep in mind.** TNet flattens the pooled feature map straight into a linear layer, so its classifier grows with the input size (16·⌊(s−2)/4⌋² inputs). Any gain therefore mixes "more pixels" with "more classifier parameters"; the parameter count is reported next to each result.


In [ ]:
# Preliminary: input resolution (only img_size changes; everything else = baseline).
SIZES = [64, 200, 224, 256]
runs = run_grid('prelim_resolution', [{'img_size': s} for s in SIZES], seeds=EXP_SEEDS)

prelim_res_results = []
for i, size in enumerate(SIZES):
    size_runs = runs[i * len(EXP_SEEDS):(i + 1) * len(EXP_SEEDS)]
    prelim_res_results.append(summarize(f'TNet @ {size}x{size}', size_runs,
                                  {'img_size': size, 'histories': [r['history'] for r in size_runs]}))

with open(RESULTS_DIR / 'prelim_resolution.json', 'w') as f:
    json.dump(prelim_res_results, f, indent=2)


In [ ]:
# Preliminary resolution run: validation curves (seed 0) for each resolution.
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for row in prelim_res_results:
    h = row['histories'][0]
    axes[0].plot(h['val_acc'], label=f"{row['img_size']}px")
    axes[1].plot(h['train_loss'], label=f"{row['img_size']}px train")
    axes[1].plot(h['val_loss'], '--', color=axes[1].lines[-1].get_color())
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Validation accuracy'); axes[0].legend()
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss (solid train, dashed val)'); axes[1].legend(fontsize=7)
plt.tight_layout()
plt.show()


In [ ]:
# FINAL TEST EVALUATION
# Run this only after you have selected your final model using validation results.
# Replace `model` with your final selected model/checkpoint.

test_loss, test_acc = evaluate(model, test_loader, device)
print(f'Final test loss: {test_loss:.4f}')
print(f'Final test accuracy: {test_acc:.4f}')

### Final model and test-set evaluation

**Selection (made on validation only, before any test evaluation).** The final model is **EfficientNet-B0 from scratch**: 200×200 grayscale (replicated to 3 channels), `hflip + rrc + translate`, Adam with a cosine schedule (peak lr 5e-3), batch 64, 200 epochs, no SSL (Experiment 4, 83.5 ± 0.5% last-10 validation). Rotation SSL was +0.8 on validation, within one std and at twice the training cost, so the simpler model is kept. We use the **final-epoch** weights, the model the last-10 metric describes, with no best-epoch picking.

The runs are re-trained here with checkpoints saved; since training is deterministic, they must reproduce the Experiment 4/6 histories exactly. Test accuracy is reported as mean ± std over the 3 seeds.

The other test numbers (rotation SSL on train; SSL on validation, Experiment 7) are **exploratory** and do not change the selection.


In [ ]:
# Final model (and the train-SSL reference) re-trained with checkpoints; must reproduce the cached runs.
from evaluate import evaluate_checkpoint
FINAL_CFG = SSL_BASE                                                   # plain EfficientNet-B0
ROT_TRAIN_CFG = {**SSL_BASE, 'ssl': 'rotation', 'ssl_weight': 1.0}     # Experiment 6, rotation λ=1
final_runs = run_grid('final', [FINAL_CFG, ROT_TRAIN_CFG], seeds=EXP_SEEDS, save_checkpoint=True)
ref = run_grid('exp4_architecture', [FINAL_CFG], seeds=EXP_SEEDS) + \
      run_grid('exp6_ssl', [ROT_TRAIN_CFG], seeds=EXP_SEEDS)
print('re-trained runs identical to Experiments 4 / 6:',
      [a['history']['val_acc'] == b['history']['val_acc'] for a, b in zip(final_runs, ref)])

def ckpts(name, cfg):
    return [RESULTS_DIR / 'runs' / name / f'{config_id(cfg)}_s{s}.pt' for s in EXP_SEEDS]

def test_summary(label, paths, val_rows=None):
    res = [evaluate_checkpoint(p, 'test') for p in paths]
    acc = np.array([r['accuracy'] for r in res]) * 100
    val = '' if val_rows is None else f"  (val last-10 {val_rows:.1f})"
    print(f"{label:34s} test {acc.mean():6.2f} ± {acc.std():4.2f}   seeds {acc.round(2).tolist()}{val}")
    return res, acc

print('\nTEST SET (400 images, 25 per class)')
final_test, final_acc = test_summary('FINAL: EfficientNet-B0 (selected)', ckpts('final', FINAL_CFG),
                                     exp6_results[0]['val_acc_last10'])
print('--- exploratory (not used for selection) ---')
test_rows = {'final': final_acc}
_, test_rows['rotation (train)'] = test_summary('rotation SSL on train, λ=1', ckpts('final', ROT_TRAIN_CFG))
for m, cfg in exp7_cfgs.items():
    _, test_rows[f'{m} (val)'] = test_summary(f'{m} SSL on val, λ={SSL_BEST_LAMBDA[m]:g}', ckpts('exp7_ssl_val', cfg))

per_class = {c: np.mean([r['per_class'][c] for r in final_test]) * 100 for c in final_test[0]['per_class']}
print('\nFinal model, per-class test accuracy (mean over seeds):')
for c, a in sorted(per_class.items(), key=lambda kv: kv[1]):
    print(f'  {c:13s} {a:5.1f}%')
with open(RESULTS_DIR / 'test_results.json', 'w') as f:
    json.dump({'final_per_class': per_class, **{k: v.tolist() for k, v in test_rows.items()}}, f, indent=2)


### Experiment 8: more labels (train+val), and FixMatch on unlabelled images

**Purpose.** Before the final scheme (supervised on train+val+test, FixMatch on the unlabelled `test2` images), this experiment checks whether the scheme helps, using `test` in the role of `test2`:

* **supervised:** all 2,400 labelled train+val images (`train_on='trainval'`);
* **unsupervised:** FixMatch on the **400 test images, labels never read** (`ssl_source='test'`);
* **evaluation:** on the same test images (labels used only here, and in a pseudo-label diagnostic that does not enter the loss).

**FixMatch.** For each unlabelled image, the model's prediction on a *weak* view (flip + RRC + translation) becomes a pseudo-label if its confidence is at least τ = 0.95. The model is then trained to predict that pseudo-label on a *strong* view (weak + RandAugment(2, 10) + Cutout). Loss = CE(labelled) + λ_u · CE(strong, pseudo-label), with λ_u = 1 and 64 labelled + 64 unlabelled images per step in one forward pass (shared batch-norm statistics). τ and λ_u are the paper's defaults and are **not tuned**: with no validation set left, any tuning could only use the test set.

**Reference.** The same train+val model **without** FixMatch, so the effect of FixMatch is separated from the effect of the 480 extra labelled images. With no validation set, nothing is selected per run: fixed 200 epochs, final-epoch weights.

**Caveats.** (1) This is transductive: the model adapts to the very images it is tested on, so its test accuracy is not comparable with the clean test number of the selected final model above. (2) Using `test2` (likely the leaderboard set) this way should be checked with the instructor. (3) A model trained on test labels has no held-out set left to report.


In [ ]:
# Experiment 8: supervised on train+val, FixMatch on the unlabelled test images; evaluated on test.
PILOT_BASE = {**SSL_BASE, 'train_on': 'trainval'}
PILOT_FM = {**PILOT_BASE, 'ssl': 'fixmatch', 'ssl_weight': 1.0, 'ssl_source': 'test', 'fixmatch_tau': 0.95}
pilot_runs = run_grid('pilot_trainval', [PILOT_BASE, PILOT_FM], seeds=EXP_SEEDS, save_checkpoint=True)

print('TEST SET (transductive for the FixMatch row)')
_, pilot_base_acc = test_summary('train+val, no unlabelled loss', ckpts('pilot_trainval', PILOT_BASE))
_, pilot_fm_acc = test_summary('train+val + FixMatch on test', ckpts('pilot_trainval', PILOT_FM))
print(f"{'reference: final model (train only)':34s} test {final_acc.mean():6.2f} ± {final_acc.std():4.2f}")

fm = pilot_runs[len(EXP_SEEDS):]
for r in fm:
    h = r['history']
    print(f"FixMatch seed {r['seed']}: mask rate (share of test images above τ) at epochs 10/50/100/200: "
          f"{[round(h['fm_mask_rate'][e - 1], 3) for e in (10, 50, 100, 200)]}; pseudo-label accuracy: "
          f"{[None if h['fm_pseudo_acc'][e - 1] is None else round(h['fm_pseudo_acc'][e - 1], 3) for e in (10, 50, 100, 200)]}")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
for r in fm:
    axes[0].plot(r['history']['fm_mask_rate'], label=f"seed {r['seed']}")
    axes[1].plot([np.nan if v is None else v for v in r['history']['fm_pseudo_acc']], label=f"seed {r['seed']}")
axes[0].set_xlabel('epoch'); axes[0].set_ylabel('share of unlabelled images above τ'); axes[0].legend(fontsize=8)
axes[1].set_xlabel('epoch'); axes[1].set_ylabel('pseudo-label accuracy (diagnostic)'); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

with open(RESULTS_DIR / 'pilot_trainval.json', 'w') as f:
    json.dump({'test_acc_no_unlabelled': pilot_base_acc.tolist(), 'test_acc_fixmatch_on_test': pilot_fm_acc.tolist(),
               'checkpoints': [str(p) for p in ckpts('pilot_trainval', PILOT_BASE) + ckpts('pilot_trainval', PILOT_FM)]},
              f, indent=2)


### Experiment 9: ImageNet-pretrained initialisation

**Question.** Does starting from ImageNet-1k weights help, compared with training EfficientNet-B0 from scratch (Experiment 4: 83.5 ± 0.5% last-10)?

**What changes.** `pretrained=True`: torchvision's `IMAGENET1K_V1` EfficientNet-B0 weights for the whole backbone; only the last linear layer is replaced (randomly initialised, 16 classes). **All layers are fine-tuned.** Inputs use the ImageNet normalisation statistics the weights were trained with (grayscale is still replicated to 3 channels). Fine-tuning usually needs a smaller learning rate, so we sweep peak lr ∈ {1e-4, 1e-3, 5e-3} and keep the best on validation. Everything else = Experiment 4 (train split only, validation for selection, 3 seeds).


In [ ]:
# Experiment 9: ImageNet-pretrained EfficientNet-B0, lr sweep (clean validation protocol).
PRE_BASE = {**SSL_BASE, 'pretrained': True}
PRE_LRS = [0.0001, 0.001, 0.005]
runs9 = run_grid('exp9_pretrained', [{**PRE_BASE, 'lr': lr} for lr in PRE_LRS], seeds=EXP_SEEDS)
exp9_results = [exp6_results[0]]   # from-scratch reference (Experiment 4)
for i, lr in enumerate(PRE_LRS):
    exp9_results.append(ssl_row(f'pretrained lr={lr:g}', runs9[i * len(EXP_SEEDS):(i + 1) * len(EXP_SEEDS)], {'lr': lr}))
print()
for row in exp9_results:
    print(f"{row['experiment']:22s} best {row['val_acc_mean']:6.2f} ± {row['val_acc_std']:4.2f}   "
          f"last-10 {row['val_acc_last10']:6.2f} ± {row['val_acc_last10_std']:4.2f}   train CE {row['final_train_loss']:.3f}")
PRE_LR = max(exp9_results[1:], key=lambda r: r['val_acc_last10'])['lr']
print('selected pretrained lr (validation):', PRE_LR)
with open(RESULTS_DIR / 'exp9_pretrained.json', 'w') as f:
    json.dump([{k: v for k, v in r.items() if k != 'histories'} for r in exp9_results], f, indent=2)


### Final recipe: clean test estimate, and the final model

The final recipe combines the best initialisation from Experiment 9 with Experiment 8's finding that more labels and FixMatch help:

* **Clean test estimate.** Supervised on train+val (2,400 images); FixMatch on the unlabelled **test2** images; evaluated on **test**. The model never sees the test images (neither labels nor pixels), so this is the honest test accuracy for the report.
* **Final model (submission).** Same recipe, but supervised on train+val+**test** (2,800 images), with FixMatch on test2 (allowed by the instructor). Nothing is held out, so it has no test score; it produces the test2 predictions.

Both use fixed 200 epochs and final-epoch weights (no early stopping or selection), and 3 seeds.


In [ ]:
# Final recipe: clean test estimate (train+val supervised, FixMatch on test2, evaluated on test).
RECIPE = {**SSL_BASE, 'pretrained': True, 'lr': PRE_LR,
          'ssl': 'fixmatch', 'ssl_weight': 1.0, 'ssl_source': 'test2', 'fixmatch_tau': 0.95}
CLEAN_CFG = {**RECIPE, 'train_on': 'trainval'}
FINAL_ALL_CFG = {**RECIPE, 'train_on': 'all'}
run_grid('final_recipe', [CLEAN_CFG, FINAL_ALL_CFG], seeds=EXP_SEEDS, save_checkpoint=True)

print('TEST SET, clean (test images never used in training):')
clean_test, clean_acc = test_summary('pretrained + FixMatch(test2), train+val', ckpts('final_recipe', CLEAN_CFG))
pc = {c: np.mean([r['per_class'][c] for r in clean_test]) * 100 for c in clean_test[0]['per_class']}
print('per-class:', ', '.join(f'{c} {a:.0f}' for c, a in sorted(pc.items(), key=lambda kv: kv[1])))
with open(RESULTS_DIR / 'final_recipe.json', 'w') as f:
    json.dump({'pretrained_lr': PRE_LR, 'clean_test_acc': clean_acc.tolist(), 'clean_test_per_class': pc,
               'final_checkpoints': [str(p) for p in ckpts('final_recipe', FINAL_ALL_CFG)]}, f, indent=2)


In [ ]:
# Final model (train+val+test supervised, FixMatch on test2): predictions for test2.
import subprocess, sys
final_ckpts = [str(p) for p in ckpts('final_recipe', FINAL_ALL_CFG)]
print(subprocess.run([sys.executable, 'predict.py', '--checkpoint', final_ckpts[0], '--images', 'data/test2',
                      '--out', 'predictions_test2.csv'], capture_output=True, text=True).stdout)
print(subprocess.run([sys.executable, 'predict.py', '--checkpoint', *final_ckpts, '--images', 'data/test2',
                      '--out', 'predictions_test2_ensemble.csv'], capture_output=True, text=True).stdout)


## 7. AI-assisted development checkpoint

Before submission, create `AI_USAGE.md` in your GitHub repository. You do **not** need to provide every prompt. Include:

1. the AI coding tool(s) you used;
2. 3--5 representative ways AI assisted you;
3. at least one AI suggestion that was incorrect, ineffective, or questionable;
4. how you tested, verified, corrected, or rejected that suggestion; and
5. one important architecture/experimental decision that **you** made.

A good entry describes the engineering/research interaction rather than saying only, ``I used ChatGPT to write code.''

### A useful habit
When AI generates code, ask yourself before running it:
- What do I expect this code to do?
- What assumption is it making about tensor shape/data/model behavior?
- How will I know if it is wrong?
- What experiment would distinguish a real improvement from noise?

## 8. GitHub and submission requirements

All assignment code must be maintained in **GitHub** (not GitLab). Your repository should contain a meaningful commit history and, at minimum:

```text
README.md
AI_USAGE.md
training / model code
inference or evaluation code
requirements.txt or environment information
final checkpoint or a link to it
reproduction instructions
```

Do not create the entire repository as a single commit immediately before the deadline. Use Git throughout development as you would in a real research or engineering project.

Your final submission is a **maximum two-page PDF report** containing the working GitHub link. The report should clearly state your best result, validation/model-selection strategy, final recipe, important experiments, at least one failure analysis, and a brief AI+human reflection.

Friendly discussion and brainstorming are encouraged, but your implementation, experiments, report, and repository must be your own. Do not share code, checkpoints, detailed hyperparameters, or test predictions before the deadline.

The instructor may ask you to explain any important component of your submitted system. You are responsible for understanding all code in your final repository, including AI-generated code.

**Deadline: September 25, 2026 at 11:59 PM EDT.**

Good luck — and treat this as your first small computer vision research project of the semester.